In [3]:
import pandas as pd
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier

# 1. เตรียมข้อมูล
df = pd.read_csv('taxonomy.csv', encoding='latin1')
features = ['Order', 'Family']
df_ml = df[features].copy().dropna()

# สร้างเป้าหมายจำลอง (Target)
def assign_habitat(order):
    if order == 'Carnivora': return 'Forest/Urban'
    if order == 'Artiodactyla': return 'Grassland'
    if order == 'Squamata': return 'Ground/Trees'
    return 'General Terrestrial'

df_ml['Target_Habitat'] = df_ml['Order'].apply(assign_habitat)

# 2. เข้ารหัสตัวแปร (Encoder)
habitat_encoders = {}
for col in features + ['Target_Habitat']:
    le = LabelEncoder()
    df_ml[col] = le.fit_transform(df_ml[col])
    habitat_encoders[col] = le

# 3. สร้างและ Train Ensemble Model
clf1 = RandomForestClassifier(n_estimators=20, random_state=42)
clf2 = GradientBoostingClassifier(n_estimators=10, random_state=42)
ensemble_habitat_model = VotingClassifier(estimators=[('rf', clf1), ('gb', clf2)], voting='soft')
ensemble_habitat_model.fit(df_ml[features], df_ml['Target_Habitat'])

# 4. เซฟโมเดลและตัวแปลงค่าลงไฟล์
with open('habitat_model.pkl', 'wb') as f:
    pickle.dump(ensemble_habitat_model, f)
with open('habitat_encoders.pkl', 'wb') as f:
    pickle.dump(habitat_encoders, f)

print("✅ เซฟโมเดลเรียบร้อย! ต่อไปจะโหลดมาใช้ได้เร็วขึ้นมาก")

✅ เซฟโมเดลเรียบร้อย! ต่อไปจะโหลดมาใช้ได้เร็วขึ้นมาก


In [4]:
import pickle
import numpy as np

# ฟังก์ชันโหลดโมเดล (แนะนำให้ใช้ @st.cache_resource ถ้าทำใน Streamlit)
def load_habitat_system():
    model = pickle.load(open('habitat_model.pkl', 'rb'))
    encoders = pickle.load(open('habitat_encoders.pkl', 'rb'))
    return model, encoders

habitat_model, h_encoders = load_habitat_system()

def predict_fast(order_name, family_name):
    try:
        # แปลงค่า
        o_enc = h_encoders['Order'].transform([order_name])[0]
        f_enc = h_encoders['Family'].transform([family_name])[0]
        
        # ทำนาย
        pred = habitat_model.predict([[o_enc, f_enc]])[0]
        result = h_encoders['Target_Habitat'].inverse_transform([pred])[0]
        return result
    except:
        return "ไม่พบข้อมูลในระบบ"

# ทดสอบรัน (จะเร็วทันที)
print("ถิ่นที่อยู่คือ:", predict_fast("Carnivora", "Felidae"))

ถิ่นที่อยู่คือ: Forest/Urban


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(
